In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from .dates import year_fraction
from .import_data import load_market, MarketEngine
from .vanilla_option import OptionType, VanillaOptionBlackSholes, d
from .binary_option import CoNBlackScholes, AoNBlackScholes

class PricingEngine:
    def __init__(self, market_file: str, tenor: str, imply_pln: bool):
        self.market = load_market(market_file, tenor)
        self.m_engine = MarketEngine(self.market)

        self.r_pln, self.r_eur, self.df_d, self.df_f = (self.m_engine.rates_dfs(imply_pln))

        self.sigma_atm = self.market.atm
        self.sigma_25C = self.market.atm + self.market.bf25 + 0.5 * self.market.rr25
        self.sigma_25P = self.market.atm + self.market.bf25 - 0.5 * self.market.rr25
        
    def strike(self,delta: float,option_type: OptionType):
        sigma = self.sigma_atm
        return strike_for_delta(
            delta=delta,
            option_type=option_type,
            df_d=self.df_d,
            df_f=self.df_f,
            S_t=self.market.spot,
            sigma=sigma,
            t=self.market.start,
            T=self.market.expiry,
            base=365,
            forward=self.market.delta_forward,
            premium=self.market.delta_premium,
        )
         
    def vanilla_price(self,K: float,option_type: OptionType,pricing_method: str,p: float = 1.0,):
        option = VanillaOptionBlackSholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def con_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001,):
        option = CoNBlackScholes(T=self.market.expiry, K=K, option_type=option_type )
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )
        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def aon_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001):
        option = AoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")







#tutaj dodac wycenę digitali





        
    def option_price_range(self, product_type:str, option_type: OptionType, K: float, pricing_method: str, n: int = 100, b: float=0.05):
        S0 = self.market.spot
        spots = np.linspace(S0 *(1-b), S0 *(1+b), n)
        prices = []

        match product_type:
            case "Vanilla":
                option = VanillaOptionBlackSholes(T=self.market.expiry,K=K,option_type=option_type)
            case "CoN":
                option = CoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
            case "AoN":
                option = AoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
            case _:
                raise ValueError(f"Nieznany typ opcji: {product_type}")
        for S in spots:
            
            if pricing_method=="bs":
                price = option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=S,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
                )
            elif pricing_method == "vv":
                if product_type=="Vanilla":
                    price = option.vanna_volga_price(
                    df_d=self.df_d,
                    df_f=self.df_f,
                    p=1.0,
                    S_t=S,
                    sigma_atm=self.sigma_atm,
                    sigma_25C=self.sigma_25C,
                    sigma_25P=self.sigma_25P,
                    t=self.market.start,
                    delta_forward=self.market.delta_forward,
                    delta_premium=self.market.delta_premium,
                    base=365,)
                elif product_type=="CoN" or product_type=="AoN":
                    price = option.vanna_volga_price(
                    df_d=self.df_d,
                    df_f=self.df_f,
                    p=1.0,
                    S_t=S,
                    sigma_atm=self.sigma_atm,
                    sigma_25C=self.sigma_25C,
                    sigma_25P=self.sigma_25P,
                    t=self.market.start,
                    delta_forward=self.market.delta_forward,
                    delta_premium=self.market.delta_premium,
                    base=365,
                    dK=0.0001
                    ) 
            else:
                raise ValueError("Metoda to nie BS ani VV")

            prices.append(price)
        return spots, np.array(prices)
    
    
    def plot_vs_spot(self,product_type:str, option_type: OptionType, K: float):
        
        spots, bs = self.option_price_range(product_type, option_type, K, "bs")
        x, vv = self.option_price_range(product_type, option_type, K, "vv")
        plt.figure()

        plt.plot(spots, bs, label="Black-Scholes")
        plt.plot(spots, vv, label="Vanna-Volga")

        plt.axvline(self.market.spot, linestyle="--", label="Spot")

        plt.xlabel("FX Spot")
        plt.ylabel("Option Price")
        plt.title(f"Cena opcji vs Spot dla {product_type} {option_type.name}")
        plt.legend()

        plt.show()

In [ ]:
engine = PricingEngine("market_data.xlsx", tenor="1Y",imply_pln=True)
vanilla = engine.vanilla_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)
print(engine.strike(0.25, OptionType.CALL))
con = engine.con_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)

aon = engine.aon_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)

print("Vanilla:", vanilla)
print("Cash-or-Nothing:", con)
print("Asset-or-Nothing:", aon)

engine.plot_vs_spot(K=engine.strike(0.25, OptionType.PUT),product_type="AoN",option_type=OptionType.CALL)